# <center>
<div style="
    background: #f0f333ff;
    border-left: 5px solid #ecd242ff;
    padding: 15px 25px;
    margin: 20px 0;
    box-shadow: 0 2px 8px rgba(0,0,0,0.08);
">
    <h1 style="
        text-align: center;
        color: #2e3a59;
        font-family: 'Segoe UI', sans-serif;
        margin: 0;
        font-weight: 600;
    ">
    Nettoyage
    </h1>
</div>

##
<div style="
    background: #40f0aa;
    border-left: 5px solid #0c7230ff;
    padding: 15px 25px;
    margin: 20px 0;
    box-shadow: 0 2px 8px rgba(0,0,0,0.08);
">
    <h2 style="
        color: #2e3a59;
        font-family: 'Segoe UI', sans-serif;
        margin: 0;
        font-weight: 500;
    ">
    Question
    </h2>
</div>

- Intégrer et tester le module fourni "validation_structure"
- Décider ce qu'est un doublon
- Appeler le service sur une vingtaine de numéros de pays diférents

###
<div style="
    background: #9feaf2ff;
    border-left: 5px solid #1d28c1ff;
    padding: 15px 25px;
    margin: 20px 0;
    box-shadow: 0 2px 8px rgba(0,0,0,0.08);
">
    <h3 style="
        color: #2e3a59;
        font-family: 'Segoe UI', sans-serif;
        margin: 0;
        font-weight: 500;
    ">
    Imports
    </h3>
</div>

In [246]:
import os
import sys
import time
import requests

import pandas as pd
import matplotlib.pyplot as plt
# On importe le module


sys.path.append('../api')
from validation_structure import normaliser, valider

# Reading Data
df_csv = pd.read_csv('../data/numeros_tva.csv', index_col='id', sep=',', keep_default_na = False)
df_xl  = pd.read_excel('../data/numeros_tva.xlsx', index_col='id', keep_default_na = False)

###
<div style="
    background: #9feaf2ff;
    border-left: 5px solid #1d28c1ff;
    padding: 15px 25px;
    margin: 20px 0;
    box-shadow: 0 2px 8px rgba(0,0,0,0.08);
">
    <h3 style="
        color: #2e3a59;
        font-family: 'Segoe UI', sans-serif;
        margin: 0;
        font-weight: 500;
    ">
    Utilisation des fonctions "Normaliser" et "Valider" du module "validation_structure.py"
    </h3>
</div>

- Suppression des caractères spéciaux
- Passage en majuscule

In [247]:
# Normalisation des numéros
df_csv['numero_norm'] = df_csv['numero_tva'].apply(normaliser)

# Validation des numéros
resultats = df_csv['numero_norm'].apply(valider)

df_csv[['Verdict', 'Motif']] = pd.DataFrame(
    resultats.tolist(),
    index=df_csv.index,
    columns=['Verdict', 'Motif']
)

df_csv.sample(10)

,raison_sociale,pays_declare,numero_tva,date_saisie,source_saisie,numero_norm,Verdict,Motif
id,,,,,,,,
1294,Kappa Transports A/S,NL,186644929B82,2024-07-10,saisie_manuelle,186644929B82,False,pays_non_couvert
6586,Alpine Trading SA,SE,SE7761453484011,2024-11-08,saisie_manuelle,SE7761453484011,False,cle_invalide
4184,Granit Materials GmbH,XX,XX0250292068,2024-04-16,crm,XX0250292068,False,pays_non_couvert
9230,Papyrus Materials SAS,LU,LU80768207,2024-05-08,portail_client,LU80768207,True,ok
5264,Verso Distribution BV,XX,XX272417932B59,2024-08-01,crm,XX272417932B59,False,pays_non_couvert
3700,Baltic Distribution Oy,LU,LU17910840,2025-07-14,portail_client,LU17910840,True,ok
6745,Nord Industries Lda,FR,null,2024-04-10,import_fournisseur,NULL,False,pays_non_couvert
1815,Solaris Transports Sp. z o.o.,FR,FR87948736522,2025-04-13,portail_client,FR87948736522,False,cle_invalide
7434,Verso Foods GmbH,FR,,2024-04-12,crm,,False,vide


Motifs possibles : 

| Motif | Explication |
|:--:|:--:|
| `ok` | Le numéro de TVA est valide |
| `vide` | Le numéro de TVA est vide |
| `trop_court` | Le numéro est trop court (moins de 4 caractères) donc invalide |
| `pays_non_couvert` | L'indicatif du pays du numéro de TAV n'est pas pris en compte |
| `cle_invalide` | La clé du numéro de TAV est invalide |

###
<div style="
    background: #9feaf2ff;
    border-left: 5px solid #1d28c1ff;
    padding: 15px 25px;
    margin: 20px 0;
    box-shadow: 0 2px 8px rgba(0,0,0,0.08);
">
    <h3 style="
        color: #2e3a59;
        font-family: 'Segoe UI', sans-serif;
        margin: 0;
        font-weight: 500;
    ">
    Comptage du nombre de numéros valides et des motifs
    </h3>
</div>

In [248]:
print('-' * 8 + ' Après avoir normalisé les numéros ' + '-' * 8 + '\n')

ds = df_csv['Verdict'].value_counts()
print(f'Nombre de numéros valides   : {ds.values[0]}')
print(f'Nombre de numéros invalides : {ds.values[1]}\n')

ds = df_csv['Motif'].value_counts()
print("Motifs des invalides :")
print(f'Nombre de numéros avec une clé invalide : {ds.values[1]}')
print(f"Nombre de numéros d'un pays non couvert : {ds.values[2]}")
print(f'Nombre de numéros vides                 : {ds.values[3]}')
print(f'Nombre de numéros trop courts           : {ds.values[4]}')
print(f'Total \t\t\t\t\t: {sum([ds.values[i] for i in range(1, 5)])}')

-------- Après avoir normalisé les numéros --------

Nombre de numéros valides   : 6083
Nombre de numéros invalides : 3917

Motifs des invalides :
Nombre de numéros avec une clé invalide : 2605
Nombre de numéros d'un pays non couvert : 1090
Nombre de numéros vides                 : 174
Nombre de numéros trop courts           : 48
Total 					: 3917


##
<div style="
    background: #40f0aa;
    border-left: 5px solid #0c7230ff;
    padding: 15px 25px;
    margin: 20px 0;
    box-shadow: 0 2px 8px rgba(0,0,0,0.08);
">
    <h2 style="
        color: #2e3a59;
        font-family: 'Segoe UI', sans-serif;
        margin: 0;
        font-weight: 500;
    ">
    Nettoyage
    </h2>
</div>


###
<div style="
    background: #9feaf2ff;
    border-left: 5px solid #1d28c1ff;
    padding: 15px 25px;
    margin: 20px 0;
    box-shadow: 0 2px 8px rgba(0,0,0,0.08);
">
    <h3 style="
        color: #2e3a59;
        font-family: 'Segoe UI', sans-serif;
        margin: 0;
        font-weight: 500;
    ">
    Sélection des pays couverts
    </h3>
</div>

On conserve uniquement les pays :<br>
"FR", "BE", "NL", "LU", "PT", "DK", "FI", "PL", "IT", "SE"

In [249]:
lst_countries = ["FR", "BE", "NL", "LU", "PT", "DK", "FI", "PL", "IT", "SE"]

print(f'Taille du dataset AVANT suppression des pays non couverts : {df_csv.shape[0]}')
df_csv = df_csv[df_csv['pays_declare'].isin(lst_countries)]
print(f'Taille du dataset APRES suppression des pays non couverts : {df_csv.shape[0]}')

Taille du dataset AVANT suppression des pays non couverts : 10000
Taille du dataset APRES suppression des pays non couverts : 9481


###
<div style="
    background: #9feaf2ff;
    border-left: 5px solid #1d28c1ff;
    padding: 15px 25px;
    margin: 20px 0;
    box-shadow: 0 2px 8px rgba(0,0,0,0.08);
">
    <h3 style="
        color: #2e3a59;
        font-family: 'Segoe UI', sans-serif;
        margin: 0;
        font-weight: 500;
    ">
    Suppression des "Faux Nan"
    </h3>
</div>

- NULL
- N/A
- NAN
- '_'

In [250]:
print(f'Taille du dataset AVANT suppression des faux NaN : {df_csv.shape[0]}')
df_csv = df_csv[~df_csv['numero_norm'].isin(['NULL', 'NA', 'NAN', ''])]
print(f'Taille du dataset APRES suppression des faux NaN : {df_csv.shape[0]}')

Taille du dataset AVANT suppression des faux NaN : 9481
Taille du dataset APRES suppression des faux NaN : 9220


###
<div style="
    background: #9feaf2ff;
    border-left: 5px solid #1d28c1ff;
    padding: 15px 25px;
    margin: 20px 0;
    box-shadow: 0 2px 8px rgba(0,0,0,0.08);
">
    <h3 style="
        color: #2e3a59;
        font-family: 'Segoe UI', sans-serif;
        margin: 0;
        font-weight: 500;
    ">
    Remplacement des O par 0
    </h3>
</div>

In [251]:
print(f"Il y a {df_csv[df_csv['numero_norm'].str.contains('O')].shape[0]} caractère 'O' dans les numéros de TVA (sur un total de {df_csv.shape[0]})")
df_csv['numero_norm'] = df_csv['numero_norm'].str.replace('O', '0', regex=False)
print(f"Il y a {df_csv[df_csv['numero_norm'].str.contains('O')].shape[0]} caractère 'O' dans les numéros de TVA (sur un total de {df_csv.shape[0]})")

Il y a 103 caractère 'O' dans les numéros de TVA (sur un total de 9220)
Il y a 0 caractère 'O' dans les numéros de TVA (sur un total de 9220)


##
<div style="
    background: #40f0aa;
    border-left: 5px solid #0c7230ff;
    padding: 15px 25px;
    margin: 20px 0;
    box-shadow: 0 2px 8px rgba(0,0,0,0.08);
">
    <h2 style="
        color: #2e3a59;
        font-family: 'Segoe UI', sans-serif;
        margin: 0;
        font-weight: 500;
    ">
    Recherche des doublons
    </h2>
</div>

Décidez ensuite de ce qu'est un doublon dans ce référentiel, et comptez-les. Votre définition doit pouvoir s'écrire en une phrase.

Nous allons chercher les doublons sur :
- Les numéros de TVA seuls
- Les couples : (numéros de TVA, raison sociale)
- Les lignes entières

####
<div style="
    background: #eebbefff;
    border-left: 5px solid #d619efff;
    padding: 15px 25px;
    margin: 20px 0;
    box-shadow: 0 2px 8px rgba(0,0,0,0.08);
">
    <h3 style="
        color: #2e3a59;
        font-family: 'Segoe UI', sans-serif;
        margin: 0;
        font-weight: 400;
    ">
    Doublon sur les lignes entières
    </h3>
</div>

In [216]:
print(f'Il y a {df_csv[df_csv.duplicated()].shape[0]} doublons pour les lignes entières')

Il y a 0 doublons pour les lignes entières


####
<div style="
    background: #eebbefff;
    border-left: 5px solid #d619efff;
    padding: 15px 25px;
    margin: 20px 0;
    box-shadow: 0 2px 8px rgba(0,0,0,0.08);
">
    <h3 style="
        color: #2e3a59;
        font-family: 'Segoe UI', sans-serif;
        margin: 0;
        font-weight: 400;
    ">
    Doublon sur les couples : (numéros de TVA, raison sociale)
    </h3>
</div>

In [217]:
# Doublon sur les numéros de TVA
df_duplicate = df_csv[df_csv[['numero_tva', 'raison_sociale']].duplicated()].sort_values(by='numero_tva')
print(f'Il y a {df_duplicate.shape[0]} doublons pour les couples (numero_tva, raison_sociale)')
display(df_duplicate.value_counts(['numero_tva', 'raison_sociale']))
df_duplicate

Il y a 132 doublons pour les couples (numero_tva, raison_sociale)


numero_tva         raison_sociale          
0282330178         Papyrus Industries SA       3
PT823046656        Kappa Foods SAS             2
  FI64257543       Fabrica Technologies A/S    1
  SE749364685001   Vela Trading SARL           1
11015857           Meridian Materials A/S      1
                                              ..
nl442736873b35     Ardent Distribution SA      1
nl785427788b39     Vela Materials SpA          1
pl1524029668       Prima Foods NV              1
pl1887742564       Lusitania Consulting Lda    1
se740563530601     Hanse Transports A/S        1
Name: count, Length: 129, dtype: int64

,raison_sociale,pays_declare,numero_tva,date_saisie,source_saisie,numero_norm,Verdict,Motif
id,,,,,,,,
8010,Fabrica Technologies A/S,FI,FI64257543,2024-08-29,portail_client,FI64257543,False,cle_invalide
5604,Vela Trading SARL,SE,SE749364685001,2024-08-08,crm,SE749364685001,True,ok
9669,Papyrus Industries SA,BE,0282330178,2024-06-05,import_fournisseur,0282330178,False,pays_non_couvert
3715,Papyrus Industries SA,BE,0282330178,2024-01-18,crm,0282330178,False,pays_non_couvert
539,Papyrus Industries SA,BE,0282330178,2025-01-16,reprise_erp,0282330178,False,pays_non_couvert
...,...,...,...,...,...,...,...,...
9852,Ardent Distribution SA,NL,nl442736873b35,2025-09-03,saisie_manuelle,NL442736873B35,True,ok
760,Vela Materials SpA,NL,nl785427788b39,2024-02-23,crm,NL785427788B39,True,ok
7822,Prima Foods NV,PL,pl1524029668,2025-04-10,saisie_manuelle,PL1524029668,True,ok


####
<div style="
    background: #eebbefff;
    border-left: 5px solid #d619efff;
    padding: 15px 25px;
    margin: 20px 0;
    box-shadow: 0 2px 8px rgba(0,0,0,0.08);
">
    <h3 style="
        color: #2e3a59;
        font-family: 'Segoe UI', sans-serif;
        margin: 0;
        font-weight: 400;
    ">
    Doublon sur les numéros de TVA seuls
    </h3>
</div>

In [218]:
df_duplicate = df_csv[df_csv['numero_tva'].duplicated()]
print(f'Il y a {df_duplicate.shape[0]} doublons pour les numéros de TVA seuls')
display(df_duplicate.sort_values(by='numero_tva'))
df_duplicate['numero_tva'].value_counts()

Il y a 132 doublons pour les numéros de TVA seuls


,raison_sociale,pays_declare,numero_tva,date_saisie,source_saisie,numero_norm,Verdict,Motif
id,,,,,,,,
8010,Fabrica Technologies A/S,FI,FI64257543,2024-08-29,portail_client,FI64257543,False,cle_invalide
5604,Vela Trading SARL,SE,SE749364685001,2024-08-08,crm,SE749364685001,True,ok
9669,Papyrus Industries SA,BE,0282330178,2024-06-05,import_fournisseur,0282330178,False,pays_non_couvert
3715,Papyrus Industries SA,BE,0282330178,2024-01-18,crm,0282330178,False,pays_non_couvert
539,Papyrus Industries SA,BE,0282330178,2025-01-16,reprise_erp,0282330178,False,pays_non_couvert
...,...,...,...,...,...,...,...,...
9852,Ardent Distribution SA,NL,nl442736873b35,2025-09-03,saisie_manuelle,NL442736873B35,True,ok
760,Vela Materials SpA,NL,nl785427788b39,2024-02-23,crm,NL785427788B39,True,ok
7822,Prima Foods NV,PL,pl1524029668,2025-04-10,saisie_manuelle,PL1524029668,True,ok


numero_tva
0282330178         3
PT823046656        2
DK73224645         1
IT 5368 0520532    1
LU75426276         1
                  ..
PL3605374689       1
ITJ5457240285      1
nl442736873b35     1
BE 1769 956921     1
FR60801776699      1
Name: count, Length: 129, dtype: int64

In [219]:
# Test un numéro en particulier
df_csv[(df_csv['numero_tva'] == '0282330178') & (df_csv['raison_sociale'] =='Papyrus Industries SA')]
df_csv[df_csv['numero_tva'] == 'ITJ5457240285']

,raison_sociale,pays_declare,numero_tva,date_saisie,source_saisie,numero_norm,Verdict,Motif
id,,,,,,,,
2265,Kappa Transports Sp. z o.o.,IT,ITJ5457240285,2024-09-18,reprise_erp,ITJ5457240285,False,cle_invalide
9834,Kappa Transports Sp. z o.o.,IT,ITJ5457240285,2025-07-31,import_fournisseur,ITJ5457240285,False,cle_invalide


**Conclusion**

Nous choisirons comme critère de "doublon" les numeros de TAV seuls car notre but est de déterminer quels numéros sont valides ou non valides et non pas de savoir si les numéros sont rattachés à la bonne société ou si la source est bonne.

In [252]:
print(f'Taille du dataset AVANT suppression des doublons : {df_csv.shape[0]}')
# df_no_duplicate = df_csv.drop_duplicates(subset=['numero_tva', 'raison_sociale'])
df_no_duplicate = df_csv.drop_duplicates(subset=['numero_tva'])
print(f'Taille du dataset APRES suppression des doublons : {df_no_duplicate.shape[0]}')

Taille du dataset AVANT suppression des doublons : 9220
Taille du dataset APRES suppression des doublons : 9088


In [253]:
print('-' * 8 + ' Après avoir normalisé les numéros sans doublon ' + '-' * 8 + '\n')
ds = df_no_duplicate['Verdict'].value_counts()
print(f'Nombre de numéros valides   : {ds.values[0]}')
print(f'Nombre de numéros invalides : {ds.values[1]}\n')

ds = df_no_duplicate['Motif'].value_counts()
display(ds)
print("Motifs des invalides :")
print(f'Nombre de numéros avec une clé invalide : {ds.values[1]}')
print(f"Nombre de numéros d'un pays non couvert : {ds.values[2]}")
print(f'Total \t\t\t\t\t: {sum([ds.values[i] for i in range(1, 3)])}')

-------- Après avoir normalisé les numéros sans doublon --------

Nombre de numéros valides   : 5998
Nombre de numéros invalides : 3090



Motif
ok                  5998
cle_invalide        2572
pays_non_couvert     518
Name: count, dtype: int64

Motifs des invalides :
Nombre de numéros avec une clé invalide : 2572
Nombre de numéros d'un pays non couvert : 518
Total 					: 3090


##
<div style="
    background: #40f0aa;
    border-left: 5px solid #0c7230ff;
    padding: 15px 25px;
    margin: 20px 0;
    box-shadow: 0 2px 8px rgba(0,0,0,0.08);
">
    <h2 style="
        color: #2e3a59;
        font-family: 'Segoe UI', sans-serif;
        margin: 0;
        font-weight: 500;
    ">
    Appel du service API (benchmark)
    </h2>
</div>

Nous allons lancer le service sur quelques numéros pour mesurer le temps de réponse.

In [ ]:
def request_api(df, size_sample):
    '''
        Requête l'API VIES (https://ec.europa.eu/taxation_customs/vies/rest-api/ms)
        Mémorise les verdicts des requêtes et les temps de réponse

        Args:
            df: DataFrame contenant les numéros de TVA
            size_sample: Nombre de numéro à requêter
        
        Returns:
            df_results : DatFrame contenant les verdicts de requêtes et les temps de réponse
            df_numero_to_check : DataFrame avec les pays+numéros qui ont donné des verdicts différents
                                 avec l'API et avec le script "valider" (cas à étudier individuellement)

    '''
    lst_sample = df[['numero_norm', 'Verdict']].sample(size_sample).values.tolist()

    lst_countries, lst_numeros, lst_script_verdict, lst_api_verdict, lst_api_time = [], [], [], [], []
    lst_numeros_to_check = []

    global_time = time.time()

    for item in lst_sample:
        code_pays, numero = item[0][:2], item[0][2:]
        start = time.time()
        try:
            r = requests.get(f"https://ec.europa.eu/taxation_customs/vies/rest-api/ms/{code_pays}/vat/{numero}", timeout=10)
            elapsed_time = time.time() - start
            if r.status_code == 200:
                lst_countries.append(code_pays)
                lst_numeros.append(numero)
                lst_script_verdict.append(item[1])
                lst_api_verdict.append("VALID" if r.json()['isValid'] else "INVALID")
                lst_api_time.append(elapsed_time)

                if r.json()['isValid'] and item[1] != 'VALID':
                    # Si l'API renvoie VALID mais que notre script renvoie INVALID
                    lst_numeros_to_check.append([code_pays, numero])

        except requests.exceptions.Timeout:
            print(f'Délai dépassé : {item[0]}')

    print(f'Temps de traitement total : {time.time() - global_time}')

    df_results = pd.DataFrame({
        'Pays' : lst_countries,
        'Numero' : lst_numeros,
        'script_verdict' : lst_script_verdict,
        'api_verdict' : lst_api_verdict,
        'api_time' : lst_api_time
    })

    df_numero_to_check = pd.DataFrame({
        'Pays'   : [item[0] for item in lst_numeros_to_check],
        'numero' : [item[1] for item in lst_numeros_to_check]
    })

    print(f'Nombre de requêtes : {size_sample}')
    print(f'Moyenne du temps de requête : {sum(lst_api_time) / size_sample}')
    return df_results, df_numero_to_check

size_sample = 10
df_results, df_numero_to_check = request_api(df_no_duplicate, size_sample)

display(df_results.sample(min(20, df_results.shape[0])))

if not(df_numero_to_check.empty):
    display(df_numero_to_check.sample(min(10, df_numero_to_check.shape[0])))

Temps de traitement total : 8.457707405090332
Nombre de requêtes : 10
Moyenne du temps de requête : 0.8457707405090332


,Pays,Numero,script_verdict,api_verdict,api_time
3,PT,986754927,True,INVALID,1.083611
2,PT,212327461,True,INVALID,1.443965
6,IT,22927333009,True,INVALID,0.896299
8,NL,293628142B12,True,INVALID,0.597434
5,DK,4108940,False,INVALID,0.502219
9,BE,0992862406,False,INVALID,0.681113
0,PL,3886374875,True,INVALID,0.954744
4,FR,34468901855,True,INVALID,0.488837
7,LU,46931518,True,INVALID,0.740941
1,IT,25990073709,True,INVALID,1.068545


In [231]:
df_csv['numero_norm'].values.tolist()

['DK73224645',
 'PT787327239',
 'LU71043234',
 'SE751298146001',
 'NL505862176B88',
 'DK61530843',
 'NL505862176B88',
 'SE314125231601',
 'DK48646344',
 'BE1997422119',
 'FI84564443',
 'NL767957817B57',
 'SE353056663101',
 'SE1861093621016',
 'PT901258090',
 'NL962988030B37',
 'BE1628475293',
 'BE0605238824',
 'FR63432337650',
 'LU45829332',
 'PT615054116',
 'FR58658124710',
 'DK73224645',
 'DK43075659',
 'BE0311256964',
 'FR14760307373',
 'SE538701827701',
 'FR84248701951',
 'PT682102423',
 'FI77720866',
 'BE0232035282',
 'DK72344081',
 'PL5047493384',
 'FI4715363',
 'FI26606369',
 'LU6918091',
 'BE124376636',
 'BE02690132662',
 'PL4321225090',
 'SE119195863401',
 'BE1290480179',
 'PT544274741',
 'SE428761352801',
 'SE011690000201',
 'NL413400931B49',
 'PL7963612947',
 'NL966200093B26',
 'SE593651753601',
 'FR82738452940',
 'LU66484248',
 'PL4372432809',
 'LU84259936',
 'BE0576561070',
 'NLH28387650B63',
 'FR76418167474',
 'FR02831068190',
 'DK53927279',
 'PT710932251',
 'PL9766889108

In [255]:
df_no_duplicate[df_no_duplicate['Motif'].str.contains("couvert")][['pays_declare', 'numero_norm']].values.tolist()

[['FR', '58560322811'],
 ['NL', '396904956B68'],
 ['IT', '38486825359'],
 ['IT', '33139885439'],
 ['IT', '88838354509'],
 ['SE', '107327005801'],
 ['PL', '5112134613'],
 ['LU', '75041556'],
 ['BE', '0572894866'],
 ['IT', '09184526433'],
 ['LU', '22312502'],
 ['LU', '13176343'],
 ['IT', '40342102486'],
 ['FR', '25167984184'],
 ['BE', '0282330178'],
 ['LU', '66781043'],
 ['IT', '92121081050'],
 ['SE', '188339040101'],
 ['PT', '264823940'],
 ['SE', '318962345101'],
 ['NL', '843131032B77'],
 ['SE', '574057769601'],
 ['IT', '63884054741'],
 ['PT', '238430596'],
 ['PT', '686019431'],
 ['NL', '160554433B57'],
 ['SE', '395554408901'],
 ['FR', '83978048922'],
 ['PL', '1151018694'],
 ['PT', '433384956'],
 ['SE', '227489464701'],
 ['DK', '48142036'],
 ['LU', '50324640'],
 ['IT', '65678084248'],
 ['BE', '1953621273'],
 ['NL', '381581160B69'],
 ['DK', '91565056'],
 ['FI', '94374459'],
 ['PL', '6723685461'],
 ['SE', '102863781501'],
 ['SE', '550629771101'],
 ['PT', '520614305'],
 ['FI', '25307340'],